# 🤖 Agentic RAG + Appointment Booking System
### Built with LangChain & LangGraph

This notebook implements a **multi-agent system** with two core capabilities:
1. 📄 **RAG Agent** — Answer questions from uploaded PDF files (Hybrid: FAISS + BM25)
2. 📅 **Appointment Agent** — Book, list, cancel, and reschedule appointments

**Architecture:**
```
User Input
    │
    ▼
Agent Node (Llama-3.3-70b via Groq)
    │
    ├──► search_pdf_knowledge ──► FAISS + BM25 Hybrid Search
    ├──► book_appointment
    ├──► list_appointments
    ├──► cancel_appointment
    ├──► get_appointment_details
    └──► reschedule_appointment
```

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    langgraph \
    faiss-cpu \
    pypdf \
    tiktoken \
    python-dotenv \
    pandas \
    openpyxl \
    rank-bm25

print("✅ All dependencies installed!")

## 🔑 Step 2: Configuration & API Keys

In [ ]:
import os
from dotenv import load_dotenv
from getpass import getpass

load_dotenv()

# ─── Set your Groq API Key ─────────────────────────────────────────────────
# Option A: Set directly (for quick testing)
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

# Option B: Load from .env file (recommended for production)
# Create a .env file with: GROQ_API_KEY=gsk_...
# Then just call load_dotenv() above

# ─── Config ────────────────────────────────────────────────────────────────
LLM_MODEL      = "llama-3.3-70b-versatile"
EMBED_MODEL    = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE     = 1000
CHUNK_OVERLAP  = 200
TOP_K_DOCS     = 4
FAISS_DIR      = "./faiss_db"
PDF_DIR        = "./pdfs"         # Put your PDFs in this folder

os.makedirs(PDF_DIR, exist_ok=True)
print("✅ Configuration ready!")

## 📚 Step 3: Core Imports

In [ ]:
import json
import uuid
from datetime import datetime, date
from typing import Annotated, Any, Literal, TypedDict

# ─── LangChain ─────────────────────────────────────────────────────────────
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# ─── Hybrid Search (with robust fallbacks) ──────────────────────────────────
try:
    try:
        from langchain_community.retrievers import BM25Retriever
    except ImportError:
        from langchain.retrievers import BM25Retriever
except ImportError as e:
    print(f"⚠️ BM25Retriever import failed: {e}")
    print("   Please ensure 'rank-bm25' is installed: pip install rank-bm25")
    BM25Retriever = None

try:
    from langchain.retrievers import EnsembleRetriever
except ImportError as e:
    print(f"⚠️ EnsembleRetriever import failed: {e}")
    EnsembleRetriever = None

# ─── LangGraph ─────────────────────────────────────────────────────────────
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

print("✅ All imports successful (with hybrid search check)!")

## 📄 Step 4: PDF Ingestion & Vector Store

In [ ]:
class PDFKnowledgeBase:
    """
    Loads PDFs → splits into chunks → embeds → stores in FAISS.
    Uses Hybrid Search (BM25 + FAISS Vector Search) via EnsembleRetriever.
    """

    def __init__(self):
        self.embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
        self.vectorstore = None
        self._initialize_vectorstore()

    def _initialize_vectorstore(self):
        """Load existing FAISS db or create a new empty one on-demand."""
        if os.path.exists(FAISS_DIR):
            try:
                self.vectorstore = FAISS.load_local(
                    FAISS_DIR,
                    self.embeddings,
                    allow_dangerous_deserialization=True
                )
                count = len(self.vectorstore.docstore._dict)
                print(f"📚 FAISS Vector store ready — {count} chunks loaded from local index.")
            except Exception as e:
                print(f"⚠️ Failed to load FAISS index: {e}. Will re-initialize on new document upload.")
                self.vectorstore = None
        else:
            self.vectorstore = None
            print("📚 FAISS Vector store ready (Empty index — will initialize on PDF upload).")

    def add_pdf(self, pdf_path: str) -> int:
        print(f"⏳ Processing: {pdf_path}")
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()

        for page in pages:
            page.metadata["source"] = os.path.basename(pdf_path)
            page.metadata["ingested_at"] = datetime.now().isoformat()

        chunks = self.text_splitter.split_documents(pages)
        if self.vectorstore is None:
            self.vectorstore = FAISS.from_documents(chunks, self.embeddings)
        else:
            self.vectorstore.add_documents(chunks)
            
        self.vectorstore.save_local(FAISS_DIR)
        print(f"   ✅ Added {len(chunks)} chunks from {len(pages)} pages to FAISS database.")
        return len(chunks)

    def add_pdf_directory(self, directory: str = PDF_DIR) -> int:
        pdf_files = [f for f in os.listdir(directory) if f.endswith(".pdf")]
        if not pdf_files:
            print(f"⚠️  No PDFs found in '{directory}'.")
            return 0
        total = 0
        for pdf in pdf_files:
            total += self.add_pdf(os.path.join(directory, pdf))
        print(f"\n🎉 Total: {total} chunks from {len(pdf_files)} PDF(s).")
        return total

    def get_retriever(self, k: int = TOP_K_DOCS):
        """Return a hybrid EnsembleRetriever combining FAISS and BM25, or fallback to FAISS."""
        if self.vectorstore is None:
            return None
            
        vector_retriever = self.vectorstore.as_retriever(search_kwargs={"k": k})
        
        # Check if hybrid search is available
        if BM25Retriever is None or EnsembleRetriever is None:
            print("⚠️ Hybrid search dependencies unavailable (missing BM25Retriever or EnsembleRetriever). Falling back to pure FAISS vector search.")
            return vector_retriever
            
        # Fetch all documents to feed BM25
        documents = list(self.vectorstore.docstore._dict.values())
        
        if documents:
            try:
                bm25_retriever = BM25Retriever.from_documents(documents)
                bm25_retriever.k = k
                # Hybrid: 50% Vector search, 50% BM25 keyword search
                return EnsembleRetriever(
                    retrievers=[vector_retriever, bm25_retriever],
                    weights=[0.5, 0.5]
                )
            except Exception as e:
                print(f"⚠️ Failed to initialize BM25Retriever ({e}). Falling back to pure FAISS vector search.")
                return vector_retriever
                
        return vector_retriever

    def similarity_search(self, query: str, k: int = TOP_K_DOCS) -> list[Document]:
        """Use hybrid retriever to search for matching documents."""
        retriever = self.get_retriever(k=k)
        if retriever is None:
            return []
        return retriever.invoke(query)


# ─── Initialize Knowledge Base ─────────────────────────────────────────────
kb = PDFKnowledgeBase()


## 📅 Step 5: Appointment Storage

In [ ]:
import pandas as pd
import uuid
import os
from datetime import datetime, date

class ExcelAppointmentStore:
    """
    Excel-based appointment store.
    Saves appointments date-wise into separate Excel files under the ./appointments/ directory.
    Enforces a maximum of 10 active appointments per day.
    Validates against past dates and duplicate time slots.
    """

    def __init__(self, base_dir="./appointments"):
        self.base_dir = base_dir
        os.makedirs(self.base_dir, exist_ok=True)

    def _get_filepath(self, date_str: str) -> str:
        try:
            parsed_date = datetime.strptime(date_str.strip(), "%Y-%m-%d").strftime("%Y-%m-%d")
        except ValueError:
            parsed_date = date_str.strip()
        return os.path.join(self.base_dir, f"appointments_{parsed_date}.xlsx")

    def _load_appointments(self, date_str: str) -> list[dict]:
        filepath = self._get_filepath(date_str)
        if not os.path.exists(filepath):
            return []
        try:
            df = pd.read_excel(filepath)
            df = df.fillna("")
            return df.to_dict(orient="records")
        except Exception:
            return []

    def _save_appointments(self, date_str: str, appts: list[dict]):
        filepath = self._get_filepath(date_str)
        df = pd.DataFrame(appts)
        df.to_excel(filepath, index=False)

    def book(self, name: str, date: str, time: str, reason: str, email: str = "") -> dict:
        try:
            clean_date = datetime.strptime(date.strip(), "%Y-%m-%d").strftime("%Y-%m-%d")
        except ValueError:
            raise ValueError(f"Invalid date format: {date}. Please use YYYY-MM-DD.")

        # ── Past-date validation ───────────────────────────────────────────
        appt_date = datetime.strptime(clean_date, "%Y-%m-%d").date()
        today = datetime.now().date()
        if appt_date < today:
            raise ValueError(
                f"Cannot book an appointment in the past ({clean_date}). "
                f"Today is {today.isoformat()}. Please choose today or a future date."
            )

        appts = self._load_appointments(clean_date)
        active_appts = [a for a in appts if a.get("status") != "cancelled"]

        # ── Daily limit check ─────────────────────────────────────────────
        if len(active_appts) >= 10:
            raise ValueError(
                f"LIMIT_EXCEEDED: The date {clean_date} is fully booked. "
                "Only 10 appointments are allowed per day."
            )

        # ── Time-slot conflict detection ──────────────────────────────────
        clean_time = time.strip()
        for existing in active_appts:
            if str(existing.get("time", "")).strip() == clean_time:
                raise ValueError(
                    f"TIME_CONFLICT: The time slot {clean_time} on {clean_date} "
                    f"is already booked by {existing.get('name', 'another patient')}. "
                    "Please choose a different time."
                )

        appt_id = str(uuid.uuid4())[:8].upper()
        appt = {
            "id": appt_id,
            "name": name,
            "date": clean_date,
            "time": clean_time,
            "reason": reason,
            "email": email,
            "status": "confirmed",
            "created_at": datetime.now().isoformat()
        }
        appts.append(appt)
        self._save_appointments(clean_date, appts)
        return appt

    def cancel(self, appt_id: str) -> dict | None:
        appt_id = appt_id.upper()
        if not os.path.exists(self.base_dir):
            return None
        for filename in os.listdir(self.base_dir):
            if filename.startswith("appointments_") and filename.endswith(".xlsx"):
                date_str = filename.replace("appointments_", "").replace(".xlsx", "")
                appts = self._load_appointments(date_str)
                for appt in appts:
                    if str(appt.get("id")).upper() == appt_id:
                        appt["status"] = "cancelled"
                        self._save_appointments(date_str, appts)
                        return appt
        return None

    def list_all(self) -> list[dict]:
        all_appts = []
        if not os.path.exists(self.base_dir):
            return []
        for filename in sorted(os.listdir(self.base_dir)):
            if filename.startswith("appointments_") and filename.endswith(".xlsx"):
                date_str = filename.replace("appointments_", "").replace(".xlsx", "")
                all_appts.extend(self._load_appointments(date_str))
        return all_appts

    def find_by_name(self, name: str) -> list[dict]:
        all_appts = self.list_all()
        return [
            a for a in all_appts
            if name.lower() in str(a.get("name", "")).lower()
        ]

    def get_by_id(self, appt_id: str) -> dict | None:
        appt_id = appt_id.upper()
        all_appts = self.list_all()
        for a in all_appts:
            if str(a.get("id")).upper() == appt_id:
                return a
        return None

# Singleton store shared across tools
appointment_store = ExcelAppointmentStore()
print("✅ Excel-based Appointment store ready (with time-slot & past-date validation)!")


## 🛠️ Step 6: Define Agent Tools

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# RAG TOOL
# ──────────────────────────────────────────────────────────────────────────

@tool
def search_pdf_knowledge(query: str) -> str:
    """
    Search the PDF knowledge base to answer questions.
    Use this tool for any question that can be answered from the uploaded documents.
    Input should be a clear, specific question or search query.
    """
    if kb.vectorstore is None:
        return "⚠️ The knowledge base is currently empty. Please load PDFs or document files to query them."
    docs = kb.similarity_search(query, k=TOP_K_DOCS)

    if not docs:
        return "No relevant information found in the PDF knowledge base."

    results = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "Unknown")
        page   = doc.metadata.get("page", "?")
        results.append(
            f"[Source {i}: {source}, Page {page}]\n{doc.page_content.strip()}"
        )

    return "\n\n---\n\n".join(results)


# ──────────────────────────────────────────────────────────────────────────
# APPOINTMENT TOOLS
# ──────────────────────────────────────────────────────────────────────────

@tool
def book_appointment(
    name: str,
    date: str,
    time: str,
    reason: str,
    email: str = ""
) -> str:
    """
    Book a new appointment for a client/patient.

    Args:
        name:   Full name of the person booking the appointment.
        date:   Date in YYYY-MM-DD format (e.g., 2025-07-15).
        time:   Time in HH:MM format, 24-hour clock (e.g., 14:30).
        reason: Brief reason or type of appointment.
        email:  Optional email address for confirmation.

    Returns a confirmation with the appointment ID.
    """
    try:
        appt = appointment_store.book(name, date, time, reason, email)
    except ValueError as e:
        return f"❌ Booking failed: {str(e)}"
    return (
        f"✅ Appointment confirmed!\n"
        f"   ID:     {appt['id']}\n"
        f"   Name:   {appt['name']}\n"
        f"   Date:   {appt['date']} at {appt['time']}\n"
        f"   Reason: {appt['reason']}\n"
        f"   Email:  {appt['email'] or 'Not provided'}\n"
        f"   Status: {appt['status']}"
    )


@tool
def list_appointments(name_filter: str = "") -> str:
    """
    List all appointments, optionally filtered by name.

    Args:
        name_filter: Optional name to filter appointments. Leave empty to show all.
    """
    if name_filter:
        appts = appointment_store.find_by_name(name_filter)
    else:
        appts = appointment_store.list_all()

    if not appts:
        return "No appointments found."

    lines = [f"📅 Found {len(appts)} appointment(s):\n"]
    for a in appts:
        status_icon = "✅" if a["status"] == "confirmed" else "❌"
        lines.append(
            f"{status_icon} [{a['id']}] {a['name']} — {a['date']} {a['time']} "
            f"| {a['reason']} | {a['status'].upper()}"
        )
    return "\n".join(lines)


@tool
def cancel_appointment(appointment_id: str) -> str:
    """
    Cancel an existing appointment by its ID.

    Args:
        appointment_id: The unique appointment ID (e.g., AB12CD34).
    """
    appt = appointment_store.cancel(appointment_id)
    if appt:
        return (
            f"❌ Appointment {appt['id']} cancelled.\n"
            f"   Name: {appt['name']} | Date: {appt['date']} {appt['time']}"
        )
    return f"⚠️  Appointment ID '{appointment_id}' not found."


@tool
def get_appointment_details(appointment_id: str) -> str:
    """
    Get full details of an appointment by ID.

    Args:
        appointment_id: The unique appointment ID.
    """
    appt = appointment_store.get_by_id(appointment_id)
    if not appt:
        return f"⚠️  No appointment found with ID '{appointment_id}'."
    return json.dumps(appt, indent=2)


@tool
def reschedule_appointment(
    appointment_id: str,
    new_date: str,
    new_time: str
) -> str:
    """
    Reschedule an existing appointment to a new date and/or time.
    This cancels the old appointment and books a new one with the same details.

    Args:
        appointment_id: The unique ID of the appointment to reschedule.
        new_date: New date in YYYY-MM-DD format.
        new_time: New time in HH:MM format, 24-hour clock.
    """
    # Find the existing appointment
    old_appt = appointment_store.get_by_id(appointment_id)
    if not old_appt:
        return f"⚠️  Appointment ID '{appointment_id}' not found. Cannot reschedule."

    if old_appt.get("status") == "cancelled":
        return f"⚠️  Appointment '{appointment_id}' is already cancelled. Please book a new one instead."

    # Cancel the old appointment
    appointment_store.cancel(appointment_id)

    # Book a new appointment with the same details but new date/time
    try:
        new_appt = appointment_store.book(
            name=old_appt["name"],
            date=new_date,
            time=new_time,
            reason=old_appt.get("reason", ""),
            email=old_appt.get("email", "")
        )
    except ValueError as e:
        # Re-activate the old appointment if rebooking fails
        # (find and un-cancel it)
        all_appts_date = appointment_store._load_appointments(old_appt["date"])
        for a in all_appts_date:
            if str(a.get("id")).upper() == appointment_id.upper():
                a["status"] = "confirmed"
                appointment_store._save_appointments(old_appt["date"], all_appts_date)
                break
        return f"❌ Reschedule failed: {str(e)}. Your original appointment remains active."

    return (
        f"🔄 Appointment rescheduled successfully!\n"
        f"   Old: {old_appt['date']} at {old_appt['time']} (cancelled)\n"
        f"   New: {new_appt['date']} at {new_appt['time']}\n"
        f"   New ID: {new_appt['id']}\n"
        f"   Name: {new_appt['name']}\n"
        f"   Status: {new_appt['status']}"
    )


# ─── Register All Tools ────────────────────────────────────────────────────
ALL_TOOLS = [
    search_pdf_knowledge,
    book_appointment,
    list_appointments,
    cancel_appointment,
    get_appointment_details,
    reschedule_appointment,
]

print(f"✅ Registered {len(ALL_TOOLS)} tools:")
for t in ALL_TOOLS:
    print(f"   • {t.name}")

## 🧠 Step 7: LangGraph Agent State & Nodes

In [ ]:
# ─── Agent State ──────────────────────────────────────────────────────────
class AgentState(TypedDict):
    """The state passed between LangGraph nodes."""
    messages: Annotated[list, add_messages]   # Full conversation history


# ─── LLM Setup ────────────────────────────────────────────────────────────
llm = ChatGroq(
    model=LLM_MODEL,
    temperature=0,
    groq_api_key=os.environ.get("GROQ_API_KEY")
)

# Bind tools to the LLM so it can call them
llm_with_tools = llm.bind_tools(ALL_TOOLS)


# ─── System Prompt ────────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are a helpful medical clinic assistant with two main capabilities:

1. **Medical Advice & PDF Knowledge Base** — You can answer questions from uploaded PDF documents.
   If a user describes medical issues, symptoms, or asks for doctor information, ALWAYS perform a search
   using the `search_pdf_knowledge` tool first.
   Provide helpful, empathetic medical advice/information from the search results, and then proactively
   offer to book an appointment for them.

2. **Appointment Management** — You can book, list, cancel, and reschedule appointments.
   - Use `book_appointment` to schedule a new appointment (collect name, date, time, reason, email).
   - Important: We only accept a maximum of 10 appointments per day. If a date is fully booked,
     suggest choosing another date.
   - Use `list_appointments` to show existing appointments.
   - Use `cancel_appointment` to cancel by ID.
   - Use `get_appointment_details` to fetch full details.
   - Use `reschedule_appointment` to move an existing appointment to a new date/time.

Guidelines:
- Always confirm appointment details with the user before booking.
- Dates must be in YYYY-MM-DD format; times in HH:MM 24-hour format.
- Do NOT book appointments in the past.
- If a question could be answered from PDFs (especially medical issues/symptoms), always search first.
- Be concise, professional, empathetic, and friendly.
- Today's date is: """ + datetime.now().strftime("%Y-%m-%d") + """\n"""


# ─── Helper: Trim conversation history to avoid context overflow ────────
MAX_HISTORY_MESSAGES = 40  # Keep last 40 messages (roughly ~20 turns)

def trim_messages(messages: list, max_messages: int = MAX_HISTORY_MESSAGES) -> list:
    """
    Trim conversation history to prevent context window overflow.
    Always keeps the first message (if it's a system message) and the last N messages.
    """
    if len(messages) <= max_messages:
        return messages
    # Keep the most recent messages
    return messages[-max_messages:]


# ─── Node: Main Agent ─────────────────────────────────────────────────────
def agent_node(state: AgentState) -> AgentState:
    """
    Core LLM agent — decides which tools to call based on the conversation.
    Includes error handling for API failures and history trimming.
    """
    trimmed = trim_messages(state["messages"])
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + trimmed

    try:
        print("🤖 [Diagnostic] Invoking LLM agent...")
        response = llm_with_tools.invoke(messages)
        if response.tool_calls:
            print(f"⚙️ [Diagnostic] LLM requested tools: {[t['name'] for t in response.tool_calls]}")
    except Exception as e:
        print(f"❌ [Error] LLM API call failed: {e}")
        error_msg = (
            "I apologize, but I am experiencing a temporary issue connecting to my AI service. "
            "Please try again in a moment. If the problem persists, check your API key and network connection."
        )
        response = AIMessage(content=error_msg)

    return {"messages": [response]}


# ─── Node: Tools ──────────────────────────────────────────────────────────
tool_node = ToolNode(ALL_TOOLS)


# ─── Edge Condition ───────────────────────────────────────────────────────
def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    """
    If the last message has tool calls → execute tools.
    Otherwise → end the turn and return response to user.
    """
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "__end__"


print("✅ Agent nodes and edges defined (router removed — LLM handles routing via tools)!")

## 🕸️ Step 8: Build the LangGraph

In [ ]:
# ─── Build Graph ──────────────────────────────────────────────────────────
workflow = StateGraph(AgentState)

# Add nodes (router removed — LLM handles intent via tool selection)
workflow.add_node("agent",  agent_node)
workflow.add_node("tools",  tool_node)

# Define edges
workflow.add_edge(START, "agent")     # Entry → Agent directly

# Conditional: Agent → Tools (if tool calls) or END
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", "__end__": END}
)

# After tools run → back to agent for final answer
workflow.add_edge("tools", "agent")

# Compile
graph = workflow.compile()

print("✅ LangGraph compiled successfully!")
print("\nGraph structure:")
print("  START → agent → [tools → agent]* → END")

## 🎨 Step 9: Visualize the Graph (Optional)

In [ ]:
# Visualize the graph structure (requires IPython)
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Graph visualization not available: {e}")
    print("\nMermaid diagram:")
    print(graph.get_graph().draw_mermaid())

## 💬 Step 10: Chat Interface

In [ ]:
class AgentChatbot:
    """Stateful chatbot wrapper around the LangGraph agent."""

    def __init__(self, graph):
        self.graph = graph
        self.conversation_history: list = []
        self.session_id = str(uuid.uuid4())[:8]
        print(f"🤖 Agent ready! Session: {self.session_id}")
        print("    Type 'quit' to exit, 'clear' to reset conversation, 'history' to view chat.\n")

    def chat(self, user_input: str) -> str:
        """Send a message and get a response."""
        if not user_input.strip():
            return ""

        # Add user message to history
        self.conversation_history.append(HumanMessage(content=user_input))

        # Trim history to prevent context window overflow
        self.conversation_history = trim_messages(self.conversation_history)

        # Run through LangGraph
        result = self.graph.invoke(
            {"messages": self.conversation_history},
            config={"recursion_limit": 20}
        )

        # Extract the final AI response
        final_messages = result["messages"]
        ai_response = ""

        # Update history with all new messages (tool calls, results, final reply)
        self.conversation_history = final_messages

        # Find last AI text message
        for msg in reversed(final_messages):
            if isinstance(msg, AIMessage) and msg.content:
                ai_response = msg.content
                break

        return ai_response

    def show_history(self):
        """Print the full conversation history."""
        print("\n" + "="*60)
        print("CONVERSATION HISTORY")
        print("="*60)
        for msg in self.conversation_history:
            if isinstance(msg, HumanMessage):
                print(f"\n👤 USER: {msg.content}")
            elif isinstance(msg, AIMessage) and msg.content:
                print(f"\n🤖 AGENT: {msg.content}")
        print("="*60 + "\n")

    def reset(self):
        """Clear conversation history."""
        self.conversation_history = []
        print("🔄 Conversation cleared.")


# Instantiate the chatbot
chatbot = AgentChatbot(graph)

## 🚀 Step 11: Interactive Chat Loop

In [ ]:
def run_interactive_chat():
    """Run an interactive terminal-style chat session."""
    print("\n" + "="*60)
    print("  🤖 RAG + APPOINTMENT BOOKING AGENT")
    print("  Powered by LangChain & LangGraph")
    print("="*60)
    print("Commands: 'quit' | 'clear' | 'history'")
    print("="*60 + "\n")

    while True:
        try:
            user_input = input("\n👤 You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Goodbye!")
            break

        if not user_input:
            continue

        if user_input.lower() == "quit":
            print("👋 Goodbye!")
            break
        elif user_input.lower() == "clear":
            chatbot.reset()
            continue
        elif user_input.lower() == "history":
            chatbot.show_history()
            continue

        response = chatbot.chat(user_input)
        print(f"\n🤖 Agent: {response}")


# ─── Run it! ──────────────────────────────────────────────────────────────
run_interactive_chat()

## 🧪 Step 12: Automated Test Suite

In [ ]:
def run_tests():
    """Run automated tests to verify the agent works end-to-end."""
    test_chatbot = AgentChatbot(graph)

    test_cases = [
        # Appointment booking
        {
            "input": "I'd like to book an appointment for Ahmed Khan on 2025-08-20 at 10:30 for a dental checkup.",
            "expect_keywords": ["confirmed", "Ahmed", "2025-08-20"],
            "label": "Book appointment"
        },
        # List appointments
        {
            "input": "Can you list all appointments?",
            "expect_keywords": ["Ahmed"],
            "label": "List appointments"
        },
        # RAG query (will return 'no info' if no PDFs loaded)
        {
            "input": "What are the main topics covered in the documents?",
            "expect_keywords": [],  # Varies based on PDFs
            "label": "PDF RAG query"
        },
        # General conversation
        {
            "input": "What can you help me with?",
            "expect_keywords": ["appointment", "pdf"],
            "label": "Capability inquiry"
        },
    ]

    print("\n" + "="*60)
    print("  🧪 RUNNING AUTOMATED TESTS")
    print("="*60)

    passed = 0
    for i, test in enumerate(test_cases, 1):
        print(f"\n[Test {i}] {test['label']}")
        print(f"  Input: {test['input'][:60]}...")

        response = test_chatbot.chat(test["input"])
        response_lower = response.lower()

        all_found = all(
            kw.lower() in response_lower
            for kw in test["expect_keywords"]
        )

        status = "✅ PASS" if (all_found or not test["expect_keywords"]) else "❌ FAIL"
        print(f"  Status: {status}")
        print(f"  Response preview: {response[:120]}...")

        if all_found or not test["expect_keywords"]:
            passed += 1

    print(f"\n{'='*60}")
    print(f"  Results: {passed}/{len(test_cases)} tests passed")
    print("="*60)


# Uncomment to run tests:
# run_tests()

## 📂 Step 13: Load Your PDFs

In [ ]:
# ─── Option A: Load all PDFs from the ./pdfs/ directory ───────────────────
# Place your PDF files in the ./pdfs/ folder, then run:
# kb.add_pdf_directory(PDF_DIR)

# ─── Option B: Load a specific PDF ───────────────────────────────────────
# kb.add_pdf("./pdfs/your_document.pdf")

# ─── Option C: Upload from Google Colab ──────────────────────────────────
def upload_pdfs_colab():
    """Upload PDFs directly in Google Colab."""
    try:
        from google.colab import files
        uploaded = files.upload()
        for filename, content in uploaded.items():
            path = os.path.join(PDF_DIR, filename)
            with open(path, "wb") as f:
                f.write(content)
            kb.add_pdf(path)
        print(f"✅ Uploaded and indexed {len(uploaded)} file(s)!")
    except ImportError:
        print("⚠️ Not running in Google Colab. Use kb.add_pdf() instead.")

# upload_pdfs_colab()  # Uncomment if using Colab

# ─── Check current vector store size ─────────────────────────────────────
count = len(kb.vectorstore.docstore._dict) if kb.vectorstore is not None else 0
print(f"📊 Vector store currently holds {count} document chunks.")
print("   Add PDFs above, then restart the chat to include them!")

## 🔧 Step 14: Single Query API (Programmatic Use)

In [ ]:
def ask(question: str, history: list = None) -> dict:
    """
    Programmatic API for the agent.

    Args:
        question: The user's message.
        history:  Optional list of previous HumanMessage/AIMessage objects.

    Returns:
        dict with 'answer' and 'messages' keys.
    """
    messages = (history or []) + [HumanMessage(content=question)]

    result = graph.invoke(
        {"messages": messages},
        config={"recursion_limit": 15}
    )

    answer = ""
    for msg in reversed(result["messages"]):
        if isinstance(msg, AIMessage) and msg.content:
            answer = msg.content
            break

    return {
        "answer":   answer,
        "messages": result["messages"]
    }


# ─── Example usage ────────────────────────────────────────────────────────

# Quick single query:
# result = ask("Book an appointment for Sara Ali on 2025-09-01 at 09:00 for consultation.")
# print(result["answer"])

# Multi-turn:
# r1 = ask("I need to book an appointment")
# r2 = ask("The name is Dr. Ziyad, date 2025-08-15, time 11:00, reason: follow-up", history=r1["messages"])
# print(r2["answer"])

print("✅ Programmatic API ready — use ask() for single queries.")

## 📊 Step 15: Architecture Summary

```
┌─────────────────────────────────────────────────────────────┐
│              AGENTIC RAG + APPOINTMENT SYSTEM               │
│                  LangChain + LangGraph                      │
└─────────────────────────────────────────────────────────────┘

 User Input
     │
     ▼
 ┌─────────┐     Tool Calls?
 │  Agent  │──────────────────┐
 │  Node   │ ← Llama-3.3-70b │  Yes
 └────┬────┘  (Groq)         ▼
      │                  ┌─────────┐
      │                  │  Tools  │
      │                  │  Node   │
      │                  └────┬────┘
      │    ┌────────────────┐ │
      │    │  search_pdf    │◄┘
      │    │  book_appt     │
      │    │  list_appts    │
      │    │  cancel_appt   │
      │    │  get_details   │
      │    │  reschedule    │
      │    └────────────────┘
      │         ↑
      │    ┌────┴──────┐   ┌───────────────┐
      │    │  FAISS    │   │ Appointment   │
      │    │ + BM25    │   │  Store (Excel)│
      │    │ (PDF RAG) │   │               │
      │    └───────────┘   └───────────────┘
      │
      ▼
 Final Answer → User
```

### Components:
| Component | Technology |
|-----------|------------|
| LLM | Llama-3.3-70b-versatile (Groq) |
| Embeddings | sentence-transformers/all-MiniLM-L6-v2 (HuggingFace) |
| Vector Store | FAISS (local index) |
| Keyword Search | BM25 via EnsembleRetriever |
| PDF Loader | PyPDFLoader (LangChain) |
| Agent Framework | LangGraph StateGraph |
| Chunking | RecursiveCharacterTextSplitter |
| Search | Hybrid (FAISS Vector Search + BM25 Keyword Search) |
| Appointments | Excel files (date-partitioned) |